In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import glob
import os
from models.aft import generate_aft_data
from matplotlib.ticker import ScalarFormatter

# 寻找所有的 aft 文件夹内的结果文件
files = glob.glob('aft/*.json')
if not files:
    print("在 aft/ 文件夹下没有找到 JSON 文件，请先运行 exp2_run_parallel_aft.ipynb")

for filename in files:
    with open(filename, 'r', encoding='utf-8') as f:
        data = json.load(f)
    res = data['results']
    params = data['parameters']
    N = len(res)
    
    print(f"================ 文件: {os.path.basename(filename)} ================")
    md_table = f"### {params.get('noise_type', 'normal')} 噪声下的性能对比 (AFT, N={N})\n"
    md_table += "| Method | RMSE | MAE | F1_Score | Pairwise | Time (s) |\n"
    md_table += "|---|---|---|---|---|---|\n"
    
    # 收集展示所有值得计算的算法
    all_methods = ['Global', 'Local', 'Avg', 'D-subGD', 'D-ProxGD', 'U-ADMM']
    all_hists = {}
    final_rmses = {}
    
    for method in all_methods:
        if method in res[0]:
            rmses = [r[method]['RMSE'] for r in res if 'RMSE' in r[method]]
            maes = [r[method]['MAE'] for r in res if 'MAE' in r[method]]
            f1s = [r[method].get('F1_Score', 0.0) for r in res]
            accs = [r[method].get('Pairwise_Correlation', 0.0) for r in res]
            times = [r[method].get('Time', 0.0) for r in res]
            
            if len(rmses) > 0:
                mean_rmse = np.mean(rmses)
                final_rmses[method] = mean_rmse
                md_table += f"| {method} | {mean_rmse:.4f} | {np.mean(maes):.4f} | {np.mean(f1s):.4f} | {np.mean(accs):.2%} | {np.mean(times):.2f} |\n"
                
                if 'hist_rmse' in res[0][method]:
                    all_hists[method] = np.mean([r[method]['hist_rmse'] for r in res], axis=0)
    
    print(md_table)
    
    
    # ============================================================
    # 1. 散点图逻辑 (分组: [U-ADMM, Global] 和 [Local, Avg])
    # ============================================================
    print("\n[绘图] 生成独立测试集散点图...")
    test_seed = params.get('rng_seed', 42) + 999 
    d_test = generate_aft_data(
        m=params['m'], n=params['n'], p_prime=params.get('p_prime', 5), 
        p=params['p'], pc=params['pc'], noise_type=params['noise_type'], 
        rng_seed=test_seed, noise_scale=params.get('noise_scale', 1.0)
    )
    X_test = d_test['X']
    theta_true = d_test['theta_true']
    true_scores = (X_test @ theta_true).flatten()
    res_one = res[0] 
    
    plot_groups = [(['U-ADMM', 'Global'], 'U-ADMM & Global'), (['Local', 'Avg'], 'Local & Avg')]
    colors = {'U-ADMM': '#3498DB', 'Global': '#FFA500', 'Local': '#2ECC71', 'Avg': '#E74C3C'}
    
    # 寻找全局统一的坐标轴范围
    score_min, score_max = true_scores.min(), true_scores.max()
    for m_key in ['U-ADMM', 'Global', 'Local', 'Avg']:
        if m_key in res_one and 'theta_hat' in res_one[m_key]:
            p_scores = (X_test @ np.array(res_one[m_key]['theta_hat']).reshape(-1, 1)).flatten()
            score_min, score_max = min(score_min, p_scores.min()), max(score_max, p_scores.max())
            
    pad = (score_max - score_min) * 0.05
    limit_range = [score_min - pad, score_max + pad]

    # 画1行2列的子图
    fig, axes = plt.subplots(1, 2, figsize=(10, 5), sharey=True)
    
    for i, (methods, title) in enumerate(plot_groups):
        ax = axes[i]
        
        for m_key in methods:
            if m_key in res_one and 'theta_hat' in res_one[m_key]:
                theta_hat = np.array(res_one[m_key]['theta_hat']).reshape(-1, 1)
                pred_scores = (X_test @ theta_hat).flatten()
                ax.scatter(true_scores, pred_scores, s=15, alpha=0.5, color=colors.get(m_key), label=m_key)
                
        ax.plot(limit_range, limit_range, 'k--', alpha=0.5, label='Ideal y=x')
        ax.set_aspect('equal', adjustable='box')
        ax.set_xlim(limit_range)
        ax.set_ylim(limit_range)
        ax.set_title(title, fontsize=12)
        ax.set_xlabel('True Latent Responses')
        if i == 0: ax.set_ylabel('Predicted Responses')
        ax.grid(True, alpha=0.2)
        ax.legend(loc='upper left')
        
    plt.tight_layout()
    plt.show()

    # ============================================================
    # 2. 合并收敛图 (只展示有迭代轨迹的算法)
    # ============================================================
    if len(all_hists) > 0 or len(final_rmses) > 0:
        print("\n[绘图] 生成合并收敛轨迹图 (所有算法放到同一张图对比)...")
        plt.figure(figsize=(9, 5.5))
        
        style_map = {
            'U-ADMM': ('-o', '#3498DB'), 
            'Global': ('-', '#FFA500'),
            'D-subGD': ('--', '#E67E22'),
            'D-ProxGD': ('--', '#9B59B6')
        }
        
        # 寻找全局最大迭代步数
        max_steps = 0
        if 'U-ADMM' in all_hists:
            max_steps = max(max_steps, (len(all_hists['U-ADMM']) - 1) * params.get('W_inner', 5))
        for m_key, hist in all_hists.items():
            if m_key != 'U-ADMM':
                max_steps = max(max_steps, len(hist) - 1)
        # 防止空轨迹时步数为0
        if max_steps == 0:
            max_steps = 100
        
        iterative_methods = ['U-ADMM', 'Global', 'D-subGD', 'D-ProxGD']
        
        # 绘制迭代算法的轨迹（如果没有轨迹但有 RMSE，则画水平直线）
        for m_key in iterative_methods:
            if m_key in final_rmses:
                color = style_map.get(m_key, ('-', 'k'))[1]
                marker = style_map.get(m_key, ('-', 'k'))[0]
                
                if m_key in all_hists:
                    # 绘制带 marker 的轨迹
                    hist = all_hists[m_key]
                    if m_key == 'U-ADMM':
                        steps = np.arange(len(hist)) * params.get('W_inner', 5)
                    else:
                        steps = np.arange(len(hist))
                    
                    plt.plot(steps, hist, marker, color=color, label=m_key, markersize=4, alpha=0.8)
                else:
                    # 缺失轨迹时的后备展示 (画一条最终 RMSE 水平线)
                    plt.hlines(final_rmses[m_key], 0, max_steps, colors=color, linestyles=':', 
                               label=f"{m_key} (RMSE={final_rmses[m_key]:.4f})")
            
        # 绘制基准水平线 (Local 和 Avg 的 RMSE 没有历史迭代)
        if 'Avg' in final_rmses:
            plt.hlines(final_rmses['Avg'], 0, max_steps, colors='#E74C3C', linestyles='--', 
                       label=f"Avg (RMSE={final_rmses['Avg']:.4f})")
        if 'Local' in final_rmses:
            plt.hlines(final_rmses['Local'], 0, max_steps, colors='#2ECC71', linestyles='-.', 
                       label=f"Local (RMSE={final_rmses['Local']:.4f})")
            
        plt.yscale('linear')
        ax = plt.gca()
        ax.yaxis.set_major_formatter(ScalarFormatter(useOffset=False))
        ax.ticklabel_format(style='plain', axis='y')
        
        plt.xlabel('Total Iterations (t)')
        plt.ylabel('RMSE')
        plt.title(f"Convergence Comparison (AFT, noise={params.get('noise_type', 'normal')})")
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.grid(True, which="both", ls="-", alpha=0.2)
        plt.tight_layout()
        plt.show()
